# MGMT298D: Science and Strategy of AI
### Week 7A - Word Embeddings
### Application: Text Representation and Similarity

## Import Libraries and Load Pre-trained Embeddings

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import requests
import zipfile
import os

np.random.seed(42)
tf.random.set_seed(42)

# Download GloVe embeddings
if not os.path.exists('glove.6B.50d.txt'):
    print("Downloading GloVe embeddings...")
    r = requests.get('http://nlp.stanford.edu/data/glove.6B.zip')
    with open('glove.6B.zip', 'wb') as f:
        f.write(r.content)
    with zipfile.ZipFile('glove.6B.zip', 'r') as zip_ref:
        zip_ref.extractall('.')

# Load embeddings
embeddings_index = {}
with open("glove.6B.50d.txt") as f:
    for line in f:
        word, coefs = line.split(maxsplit=1)
        coefs = np.fromstring(coefs, "f", sep=" ")
        embeddings_index[word] = coefs
        
print(f"Loaded {len(embeddings_index):,} word vectors")

## Visualize Word Relationships

In [ ]:
def plot_embeddings(words):
    vectors = np.array([embeddings_index[word] for word in words])
    pca = PCA(n_components=2)
    vectors_2d = pca.fit_transform(vectors)
    
    plt.figure(figsize=(10, 8))
    plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], color='blue')
    for i, word in enumerate(words):
        plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), fontsize=12)
    plt.title("Word Embeddings in 2D Space")
    plt.grid(True)
    plt.show()

words_to_plot = ['king', 'queen', 'man', 'woman', 'prince', 'princess', 
                 'dog', 'cat', 'banana', 'apple', 'car', 'truck']
plot_embeddings(words_to_plot)

## Learn Embeddings from Scratch: Skip-gram Model

In [ ]:
from keras.preprocessing.text import Tokenizer

# Small corpus for demonstration
corpus = [
    'the quick brown fox jumps over the lazy dog',
    'the king and queen ruled the kingdom',
    'a man and a woman walked down the street',
    'the prince is the son of the king'
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
sequences = tokenizer.texts_to_sequences(corpus)

vocab_size = len(tokenizer.word_index) + 1
word_index = tokenizer.word_index
print(f"Vocabulary size: {vocab_size}")

# Generate skip-gram pairs
skip_grams = [keras.preprocessing.sequence.skipgrams(seq, vocabulary_size=vocab_size, window_size=5) for seq in sequences]

pairs, labels = [], []
for el in skip_grams:
    pairs.extend(el[0])
    labels.extend(el[1])

target_word, context_word = zip(*pairs)
target_word = np.array(target_word)
context_word = np.array(context_word)
labels = np.array(labels)

print(f"Generated {len(labels):,} skip-gram pairs")

In [ ]:
# Build Skip-gram model
EMBED_DIM = 50

input_target = layers.Input((1,))
input_context = layers.Input((1,))

embedding = layers.Embedding(vocab_size, EMBED_DIM, name="embedding_layer")

target_vector = embedding(input_target)
target_vector = layers.Reshape((EMBED_DIM, 1))(target_vector)

context_vector = embedding(input_context)
context_vector = layers.Reshape((EMBED_DIM, 1))(context_vector)

dot_product = layers.Dot(axes=1)([target_vector, context_vector])
dot_product = layers.Reshape((1,))(dot_product)
output = layers.Dense(1, activation='sigmoid')(dot_product)

skipgram_model = keras.Model(inputs=[input_target, input_context], outputs=output)
skipgram_model.compile(loss='binary_crossentropy', optimizer='adam')

skipgram_model.fit([target_word, context_word], labels, epochs=50, verbose=1)

learned_embeddings = skipgram_model.get_layer("embedding_layer").get_weights()[0]
print(f"\nLearned vector for 'king' (first 5 dims): {learned_embeddings[word_index['king']][:5]}")

## Apply Pre-trained Embeddings: Sentiment Classification

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 250
EMBED_DIM = 50

# Load IMDb dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)
x_train = keras.utils.pad_sequences(x_train, maxlen=MAX_LEN)
x_test = keras.utils.pad_sequences(x_test, maxlen=MAX_LEN)

# Create embedding matrix from GloVe
word_index_imdb = keras.datasets.imdb.get_word_index()
num_tokens = min(VOCAB_SIZE, len(word_index_imdb) + 1)
embedding_matrix = np.zeros((num_tokens, EMBED_DIM))

for word, i in word_index_imdb.items():
    if i >= VOCAB_SIZE:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

print(f"Embedding matrix shape: {embedding_matrix.shape}")

In [ ]:
# Build classifier with pre-trained embeddings
embedding_layer = layers.Embedding(
    num_tokens, EMBED_DIM,
    embeddings_initializer=keras.initializers.Constant(embedding_matrix),
    trainable=False  # Freeze embeddings
)

inputs = layers.Input(shape=(MAX_LEN,))
x = embedding_layer(inputs)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(32, activation='relu')(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    x_train, y_train, batch_size=128, epochs=10, validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)]
)

loss, acc = model.evaluate(x_test, y_test)
print(f"\nSentiment Classification — Test Accuracy: {acc*100:.2f}%")

## Word Analogies: Vector Arithmetic

In [ ]:
# Prepare normalized vectors for similarity search
all_words, all_vectors = zip(*embeddings_index.items())
all_vectors = np.array(all_vectors)
all_vectors_norm = all_vectors / np.linalg.norm(all_vectors, axis=1, keepdims=True)
word_to_vec = {word: vec for word, vec in embeddings_index.items()}

def find_most_similar(vec):
    vec_norm = vec / np.linalg.norm(vec)
    sim = np.dot(all_vectors_norm, vec_norm.T)
    top_indices = np.argsort(sim)[-6:-1][::-1]
    return [all_words[i] for i in top_indices]

def solve_analogy(word1, word2, word3):
    """word1 - word2 + word3 = ?"""
    try:
        result_vec = word_to_vec[word1] - word_to_vec[word2] + word_to_vec[word3]
        return find_most_similar(result_vec)[0]
    except KeyError as e:
        return f"'{e.args[0]}' not in vocabulary"

# Test analogies
print("Analogy: 'king' - 'man' + 'woman' =", solve_analogy('king', 'man', 'woman'))
print("Analogy: 'paris' - 'france' + 'germany' =", solve_analogy('paris', 'france', 'germany'))
print("Analogy: 'walking' - 'walk' + 'swim' =", solve_analogy('walking', 'walk', 'swim'))

print("\nWords similar to 'music':", find_most_similar(word_to_vec['music']))